In [9]:
#!/usr/bin/env python3
"""
ner_bibliography.py
────────────────────────────────────────────────────────────────────
Regex-based NER for Italian fiscal-history bibliography pages.

Entry format (as produced by Gemini OCR, with markdown):
    NNNN

    **Italian title** (*Original title*)[, di Author Name].
    — *Journal Name*, [DD] Month YYYY, pagg./pagine X-Y.

    <abstract text ...>

Usage:
    python ner_bibliography.py            # extract → ocr_output_gemini.json
    python ner_bibliography.py diagnose   # report unmatched numeric lines
"""

import re
import json
import sys
from pathlib import Path

# ──────────────────────────────────────────────────────────────────
# COMPILED PATTERNS
# ──────────────────────────────────────────────────────────────────

RE_ENTRY_NUM = re.compile(r'^\s*\*{0,2}(\d{4})\*{0,2}\s*$', re.MULTILINE)

# Strip markdown bold (**text**) and italic (*text*)
RE_BOLD   = re.compile(r'\*\*(.+?)\*\*')
RE_ITALIC = re.compile(r'\*(.+?)\*')

# Source line: — Journal, [DD] Month YYYY, pagg./pagine X-Y
# Journal may be wrapped in *italics* by Gemini — stripped before matching,
# but the em-dash may also be followed directly by *Journal*
RE_SOURCE = re.compile(
    r'[—–\-]\s*(.+?),\s*'
    r'(\d{1,2}\s+\w+[\w-]*\s+\d{4}|\w+[\w-]*\s+\d{4}),\s*'
    r'(?:pagg?\.|pagine?)\s*([\d\-–]+)',
    re.IGNORECASE
)

# Author: ", di Name" — allows optional newline between "di" and name
RE_AUTHOR = re.compile(
    r',\s*di\s*\n?\s*'
    r'([A-ZÀÈÉÌÒÙ][a-zàèéìòùë]+(?:\s+[A-ZÀÈÉÌÒÙ][a-zàèéìòùë]+)+'
    r'|[A-Z]\.\s*[A-ZÀÈÉÌÒÙ][a-zàèéìòùë]+'
    r'|[A-Z]\.[A-Z]\.\s*[A-ZÀÈÉÌÒÙ][a-zàèéìòùë]+)',   # e.g. C.L. Harris
    re.MULTILINE
)

# First blank line after the source line marks start of abstract
RE_ABSTRACT_START = re.compile(r'\n\s*\n')


# ──────────────────────────────────────────────────────────────────
# MARKDOWN STRIPPING
# ──────────────────────────────────────────────────────────────────

def strip_markdown(text: str) -> str:
    """Remove bold (**text**) and italic (*text*) markers, keeping inner text."""
    text = RE_BOLD.sub(r'\1', text)
    text = RE_ITALIC.sub(r'\1', text)
    return text


# ──────────────────────────────────────────────────────────────────
# HELPERS
# ──────────────────────────────────────────────────────────────────

def split_into_entries(text: str) -> list[tuple[str, str]]:
    """Split full OCR text into (entry_id, raw_block) pairs."""
    parts = RE_ENTRY_NUM.split(text)
    return [
        (parts[i].strip(), parts[i + 1].strip())
        for i in range(1, len(parts) - 1, 2)
    ]


def split_meta_from_abstract(block: str) -> tuple[str, str]:
    """
    Separate metadata lines (title + author + source) from abstract body.
    Anchors the split to after the source line to avoid cutting at blank
    lines that may appear between the title and the source line.
    """
    src_m = RE_SOURCE.search(block)
    if src_m:
        after_src = src_m.end()
        blank_m = RE_ABSTRACT_START.search(block, after_src)
        cut = blank_m.start() if blank_m else len(block)
        return block[:cut].strip(), block[cut:].strip()
    blank_m = RE_ABSTRACT_START.search(block)
    if blank_m:
        return block[:blank_m.start()].strip(), block[blank_m.end():].strip()
    return block.strip(), ''


def parse_meta_block(meta: str) -> tuple:
    """
    Parse metadata block → (title_it, title_original, author, journal, date, pages).
    Strips markdown before parsing so **bold** and *italic* don't interfere.
    """
    # Strip markdown from the whole meta block first
    meta_clean = strip_markdown(meta)

    # 1. Author
    author = None
    author_m = RE_AUTHOR.search(meta_clean)
    if author_m:
        author = re.sub(r'\s+', ' ', author_m.group(1)).strip()
        meta_for_title = meta_clean[:author_m.start()]
    else:
        meta_for_title = meta_clean

    # 2. Source line
    journal = date = pages = None
    src_m = RE_SOURCE.search(meta_clean)
    if src_m:
        journal = src_m.group(1).strip()
        date    = src_m.group(2).strip()
        pages   = src_m.group(3).strip()
        title_raw = meta_for_title[:src_m.start()].strip() if not author_m \
                    else meta_for_title.strip()
    else:
        title_raw = meta_for_title.split('\n')[0].strip()

    # 3. Italian title + original title
    title_raw = re.sub(r',\s*di\s*$', '', title_raw, flags=re.IGNORECASE).strip()
    title_raw = re.sub(r'\s+', ' ', title_raw)

    paren_m = re.search(r'\(([^)]+)\)\s*\.?\s*$', title_raw)
    if paren_m:
        title_original = re.sub(r'\s+', ' ', paren_m.group(1)).strip()
        title_it = title_raw[:paren_m.start()].strip().rstrip(' .,')
    else:
        title_original = None
        title_it = title_raw.rstrip(' .,')

    return title_it or None, title_original, author, journal, date, pages


def parse_entry(entry_id: str, block: str) -> dict:
    meta, _abstract = split_meta_from_abstract(block)
    title_it, title_original, author, journal, date, pages = parse_meta_block(meta)
    return {
        "entry_id":       entry_id,
        "title_it":       title_it,
        "title_original": title_original,
        "author":         author,
        "journal":        journal,
        "date":           date,
        "pages":          pages,
    }


def extract_from_text(text: str) -> list[dict]:
    return [parse_entry(num, block) for num, block in split_into_entries(text)]


def diagnose(text: str) -> None:
    """
    Print every line that looks like a bare number but was NOT matched
    as a valid 4-digit entry number, to help identify OCR misreads.
    """
    matched = {m.group(1) for m in RE_ENTRY_NUM.finditer(text)}
    RE_CANDIDATE = re.compile(r'^\s*(\d{3,6})\s*$', re.MULTILINE)
    missed = [
        m.group(1) for m in RE_CANDIDATE.finditer(text)
        if m.group(1) not in matched
    ]
    print(f"Matched entry numbers ({len(matched)}): {sorted(matched)}")
    print(f"Unmatched numeric lines ({len(missed)}):  {missed}")



# ──────────────────────────────────────────────────────────────────
# CLI
# ──────────────────────────────────────────────────────────────────

INPUT_FILE  = Path("ocr_output_gemini.txt")
OUTPUT_FILE = Path("ner_regex.json")

if __name__ == "__main__":
    if len(sys.argv) > 1 and sys.argv[1] == "test":
        print("Running self-test on sample page data…\n")
        self_test()
    else:
        text = INPUT_FILE.read_text(encoding="utf-8")
        results = extract_from_text(text)
        OUTPUT_FILE.write_text(
            json.dumps(results, ensure_ascii=False, indent=2),
            encoding="utf-8"
        )
        print(f"Extracted {len(results)} entries → {OUTPUT_FILE}")

Extracted 800 entries → ner_regex.json
